# Solutions to project 4: every muon in every event, with Awkward Array

Projects 2 and 3 only used events with exactly two electrons and two muons, because NumPy needs every row to have the same length.
Here you use every event, with however many muons it has.

<img src="../img/jagged.svg" width="70%" alt="">

**How to work**

- Run each cell with `Shift` + `Enter`, from top to bottom.
- Replace every `...` with your own code.
- Each part ends with a `check`. ✅ means move on. ❌ comes with a hint.
- Solutions are in the `solutions` folder. Try for five minutes before you look.

In [ ]:
import sys

sys.path.insert(0, "..")  # so that checks.py is found

import awkward as ak
import hist
import matplotlib.pyplot as plt
import numpy as np
import vector

from checks import check

vector.register_awkward()
events = ak.from_parquet("../data/SMHiggsToZZTo4L.parquet")
events.muon.pt

## Part 1: Counting muons

`ak.num` counts the muons in every event.

**Task.** How many events have exactly two muons? `ak.sum` adds up the `True` values of a mask.

In [ ]:
n_two = ak.sum(ak.num(events.muon) == 2)
check("4.1", n_two)

## Part 2: Arithmetic

Arithmetic works as in NumPy.

**Task.** Compute $p_z = p_T \sinh\eta$ for every muon in every event.

In [ ]:
pz = events.muon.pt * np.sinh(events.muon.eta)
check("4.2", pz)

## Part 3: Reductions

A reduction with `axis=1` gives one value per event.

**Task.** Find the largest muon $p_T$ in each event.
Events without muons give `None`. `ak.drop_none` removes them before plotting.

<img src="../img/reduce-max.svg" width="70%" alt="">

In [ ]:
leading = ak.max(events.muon.pt, axis=1)
check("4.3", leading)

In [ ]:
h = hist.Hist.new.Reg(50, 0, 150, label="leading muon pT [GeV]").Double()
h.fill(ak.drop_none(leading))
h.plot();

## Part 4: Selections

Select muons, then select events.

Keep the muons with $p_T > 5$ GeV and $|\eta| < 2.4$.
Then keep the events with exactly two of them.

**Task.** The first line is done for you. Write the event selection.

<img src="../img/two-masks.svg" width="80%" alt="">

In [ ]:
muons = events.muon[(events.muon.pt > 5) & (np.abs(events.muon.eta) < 2.4)]
two_muons = muons[ak.num(muons) == 2]
check("4.4", two_muons)

## Part 5: Two muons

With exactly two muons, `[:, 0]` and `[:, 1]` are safe.

`vector` adds two muons with `+` and gives the mass with `.mass`, like your `FourVector` class.

**Task.** Keep the pairs with opposite charge and compute their mass.

<img src="../img/two-muons.svg" width="60%" alt="">

In [ ]:
mu1 = two_muons[:, 0]
mu2 = two_muons[:, 1]
dimuon = (mu1 + mu2)[mu1.charge != mu2.charge]
check("4.5", dimuon)

In [ ]:
h = hist.Hist.new.Reg(60, 0, 120, label="dimuon mass [GeV]").Double()
h.fill(dimuon.mass)
h.plot();

## Part 6: Every pair

`ak.combinations` makes every pair, whatever the number of muons.

**Task.** Make every pair of the selected `muons` in every event, keep the opposite-charge pairs, and compute their masses.
The masses are jagged, one per pair, so flatten them before filling the histogram.

<img src="../img/combinations.svg" width="70%" alt="">

In [ ]:
mu1, mu2 = ak.unzip(ak.combinations(muons, 2))
pairs = (mu1 + mu2)[mu1.charge != mu2.charge]
check("4.6", pairs)

In [ ]:
h_pairs = hist.Hist.new.Reg(60, 0, 120, label="dimuon mass [GeV]").Double()
h_pairs.fill(ak.flatten(pairs.mass))
h.plot(label="events with exactly two muons")
h_pairs.plot(label="every opposite-charge pair")
plt.legend();

## Bonus: Back to project 2

You get the same answer as project 2.

**Task.** Select the events with exactly two electrons and two muons, keep those where both pairs have opposite charge, and compute the four-lepton mass.
Project 2 found 27,496 such events.

In [ ]:
eemumu = events[(ak.num(events.electron) == 2) & (ak.num(events.muon) == 2)]
e1, e2 = eemumu.electron[:, 0], eemumu.electron[:, 1]
m1, m2 = eemumu.muon[:, 0], eemumu.muon[:, 1]
opposite = (e1.charge != e2.charge) & (m1.charge != m2.charge)
higgs_masses = (e1 + e2 + m1 + m2).mass[opposite]
check("4.7", higgs_masses)

**Done.** The next lectures in the school continue with Awkward Array and a full HEP analysis, leading to coffea.